In [ ]:
# %pip install peft evaluate
# %pip install -U torch torchaudio torchcodec torchao mslk --index-url https://download.pytorch.org/whl/cu128

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU device: {torch.cuda.get_device_name(0)}")

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
except:
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception as e:
        print(e)

In [ ]:
labels = {
    0: "AG",
    1: "BE",
    2: "BS",
    3: "GR",
    4: "LU",
    5: "SG",
    6: "VS",
    7: "ZH",
}

id2label = labels
label2id = {v: k for k, v in labels.items()}
num_labels = len(labels)

In [ ]:
from transformers import Wav2Vec2Config, Wav2Vec2ForSequenceClassification, AutoFeatureExtractor, Wav2Vec2Processor
import torch

model_id = "facebook/wav2vec2-xls-r-300m"

#processor = WhisperProcessor.from_pretrained(model_id)
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)

config = Wav2Vec2Config.from_pretrained(
    model_id,
    num_labels=8,
    label2id=label2id,
    id2label=id2label,
    # SpecAugment parameters
    apply_spec_augment=True,
    mask_time_prob=0.05,
    mask_time_length=10,
    mask_feature_prob=0.05,
    mask_feature_length=10,
)

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    model_id,
    config=config
)

model.freeze_feature_encoder()

In [ ]:
from peft import PeftModel, LoraConfig, get_peft_model, TaskType
from peft.optimizers import create_lorafa_optimizer

peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        "q_proj",
        "v_proj",
        "k_proj",
        "out_proj",
        "intermediate.dense",
        "output.dense",
    ],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
import datasets
from datasets import load_dataset, load_from_disk, Audio, interleave_datasets, concatenate_datasets, Value

# Training data: Combine SwissDial and ArchiMob and balance by region
ds_swissdial = load_dataset("RobChio/swiss-dial-preprocessed", split="train")
ds_archimob_train = load_dataset("RobChio/archimob-preprocessed", split="train")

ds_swissdial = ds_swissdial.cast_column("dialect_code", Value(dtype="int64")) # fix type mismatch

ds_swissdial.set_format(type="torch", columns=["audio", "dialect_code"])
ds_archimob_train.set_format(type="torch", columns=["audio", "dialect_code"])

combined_train = concatenate_datasets([ds_swissdial, ds_archimob_train])
train_dataset = combined_train.shuffle(seed=42)
#train_dataset = balance_dataset(combined_train, label_col="dialect_code", seed=42)
train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16000))

# Validation data from ArchiMob
ds_archimob_val = load_dataset("RobChio/archimob-preprocessed", split="validation")
val_dataset = ds_archimob_val.shuffle(seed=42)
val_dataset.set_format(type="torch", columns=["audio", "dialect_code"])
val_dataset = val_dataset.cast_column("audio", Audio(sampling_rate=16000))

In [ ]:
def preprocess_function(batch):
    audios = [x["array"] for x in batch["audio"]]
    inputs = feature_extractor(
        audios,
        sampling_rate=16000,
        max_length=160000,  # Truncate at 10 seconds (16,000 Hz * 10s)
        truncation=True,
    )
    inputs["label"] = batch["dialect_code"]
    return inputs

# Apply mapping (audio is loaded on the fly)
train_dataset_preprocessed = train_dataset.map(
    preprocess_function,
    remove_columns=["audio"],
    batched=True,
    batch_size=64,
    num_proc=16,
)

val_dataset_preprocessed = val_dataset.map(
    preprocess_function,
    remove_columns=["audio"],
    batched=True,
    batch_size=64,
    num_proc=16,
)

In [ ]:
import torch
from torch import nn
from transformers import Trainer
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        if class_weights is not None and not isinstance(
            class_weights, torch.Tensor
        ):
            self.class_weights = torch.tensor(class_weights, dtype=torch.float32)
        else:
            self.class_weights = class_weights
            
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        smoothing = getattr(self.args, "label_smoothing_factor", 0.0)

        if self.class_weights is not None:
            weights = self.class_weights.to(logits.device)
            loss_fct = nn.CrossEntropyLoss(weight=weights, label_smoothing=smoothing)
        else:
            loss_fct = nn.CrossEntropyLoss(label_smoothing=smoothing)

        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [ ]:
train_labels = train_dataset_preprocessed.with_format("numpy")["label"]
# train_labels = train_dataset_preprocessed["label"]

class_weights = compute_class_weight(
    class_weight="balanced", classes=np.arange(8), y=train_labels
)

In [ ]:
from torch.nn.utils.rnn import pad_sequence

class FastClassificationDataCollator:
    def __call__(self, features):
        tensors = [
            torch.as_tensor(f["input_values"], dtype=torch.float32) 
            for f in features
        ]
        # Dynamic CPU padding to the longest sequence in the current batch
        input_values = pad_sequence(tensors, batch_first=True, padding_value=0.0)
        labels = torch.tensor([f["label"] for f in features], dtype=torch.long)
        return {"input_values": input_values, "labels": labels}

data_collator = FastClassificationDataCollator()

In [ ]:
from transformers import TrainingArguments, Trainer, get_cosine_schedule_with_warmup
import evaluate
import numpy as np
import torch.nn.functional as F

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    predictions = np.argmax(logits, axis=-1)

    macro_f1 = f1.compute(predictions=predictions, references=labels, average="macro")["f1"]
    acc = accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    return {"f1": macro_f1, "accuracy": acc}

training_args = TrainingArguments(
    output_dir="./swissgerman-dialect-classifier-xls-r",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-4,
    label_smoothing_factor=0.1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    warmup_steps=500,
    lr_scheduler_type="cosine",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    remove_unused_columns=False,
    dataloader_num_workers=16,
    dataloader_pin_memory=True,
    dataloader_persistent_workers=True,
    dataloader_prefetch_factor=4,
    torch_compile=False,
    dataloader_drop_last=False,
    push_to_hub=False,
    hub_model_id="RobChio/swissgerman-dialect-classifier-xls-r",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_preprocessed,
    eval_dataset=val_dataset_preprocessed,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

In [ ]:
train_result = trainer.train()

In [ ]:
trainer.push_to_hub()

In [ ]:
# from google.colab import runtime
# runtime.unassign()